# Simulação da Copa do Mundo 2026: Fase de Grupos
Neste notebook, calculamos os parâmetros de força (ataque e defesa) das seleções para a Copa do Mundo usando dois modelos: **Ingênuo (Poisson)** e **Dixon-Coles**.
Em seguida, armazenamos esses parâmetros em um CSV (nosso novo banco de dados) para que o cálculo seja feito apenas uma vez.
Por fim, simulamos de forma clara e didática todos os jogos da fase de grupos!

*Nota:* Os dados foram filtrados para jogos a partir de **01/01/2024** e o parâmetro de decaimento de tempo no modelo Dixon-Coles foi ajustado para **xi = 0.0065**.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize
import datetime
import warnings
import json
warnings.filterwarnings('ignore')

# 1. Carregar os dados e filtrar
df = pd.read_csv('results.csv')
df['date'] = pd.to_datetime(df['date'])

# FILTRO ATUALIZADO AQUI (a partir de 01/01/2024)
df = df[(df['date'] >= '2024-01-01')]

df = df.dropna(subset=['home_score', 'away_score'])
df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

print(f"Total de jogos analisados a partir de 2024: {len(df)}")

# Seleções participantes (Projeção de 48 seleções para 2026)
selecoes_copa = [
    'Mexico', 'South Korea', 'Czech Republic', 'South Africa',
    'Canada', 'Switzerland', 'Bosnia and Herzegovina', 'Qatar',
    'Brazil', 'Morocco', 'Scotland', 'Haiti',
    'United States', 'Australia', 'Paraguay', 'Turkey',
    'Germany', 'Ivory Coast', 'Ecuador', 'Curaçao',
    'Netherlands', 'Japan', 'Sweden', 'Tunisia',
    'Egypt', 'Iran', 'Belgium', 'New Zealand',
    'Spain', 'Uruguay', 'Cape Verde', 'Saudi Arabia',
    'France', 'Norway', 'Senegal', 'Iraq',
    'Argentina', 'Austria', 'Algeria', 'Jordan',
    'Portugal', 'Colombia', 'DR Congo', 'Uzbekistan',
    'England', 'Ghana', 'Panama', 'Croatia'
]

# Filtrar para considerar apenas seleções que têm dados e ignorar o resto
times_gerais = df['home_team'].value_counts()[df['home_team'].value_counts() >= 1].index
df_geral = df[df['home_team'].isin(times_gerais) & df['away_team'].isin(times_gerais)]

# 2. Modelo Ingênuo
media_gols_geral = (df_geral['home_score'].mean() + df_geral['away_score'].mean()) / 2
ataque_ingenuo = {}
defesa_ingenua = {}

for sel in selecoes_copa:
    jogos_casa = df_geral[df_geral['home_team'] == sel]
    jogos_fora = df_geral[df_geral['away_team'] == sel]
    total = len(jogos_casa) + len(jogos_fora)
    if total > 0:
        gf = jogos_casa['home_score'].sum() + jogos_fora['away_score'].sum()
        gs = jogos_casa['away_score'].sum() + jogos_fora['home_score'].sum()
        ataque_ingenuo[sel] = (gf / total) / media_gols_geral
        defesa_ingenua[sel] = (gs / total) / media_gols_geral
    else:
        ataque_ingenuo[sel] = 1.0 # default caso não tenha jogos
        defesa_ingenua[sel] = 1.0

# 3. Modelo Dixon-Coles
selecoes = list(set(df_geral['home_team'].unique()) | set(df_geral['away_team'].unique()))
n_selecoes = len(selecoes)
sel_idx = {sel: i for i, sel in enumerate(selecoes)}

df_opt = df_geral.copy()
df_opt['home_idx'] = df_opt['home_team'].map(sel_idx)
df_opt['away_idx'] = df_opt['away_team'].map(sel_idx)
hoje = pd.to_datetime('today')
df_opt['dias_atras'] = (hoje - df_opt['date']).dt.days
df_opt['neutral'] = df_opt['neutral'].fillna(False)

# PARÂMETRO XI ATUALIZADO AQUI
xi = 0.0065

def dixon_coles_loglik_vectorized(params, df_matches, xi, lambda_reg=0.05):
    alpha = params[:n_selecoes]
    beta = params[n_selecoes:2*n_selecoes]
    rho = params[2*n_selecoes]
    gamma = params[2*n_selecoes + 1]
    
    idx_home = df_matches['home_idx'].values
    idx_away = df_matches['away_idx'].values
    x = df_matches['home_score'].values
    y = df_matches['away_score'].values
    dias = df_matches['dias_atras'].values
    is_neutral = df_matches['neutral'].values
    
    gamma_adj = np.where(is_neutral, 0.0, gamma)
    
    lam = np.exp(alpha[idx_home] + beta[idx_away] + gamma_adj)
    mu = np.exp(alpha[idx_away] + beta[idx_home])
    pesos = np.exp(-xi * dias)
    
    prob_poisson_x = poisson.pmf(x, lam)
    prob_poisson_y = poisson.pmf(y, mu)
    
    corr = np.ones_like(x, dtype=float)
    corr[(x == 0) & (y == 0)] = 1 - (lam[(x == 0) & (y == 0)] * mu[(x == 0) & (y == 0)] * rho)
    corr[(x == 0) & (y == 1)] = 1 + (lam[(x == 0) & (y == 1)] * rho)
    corr[(x == 1) & (y == 0)] = 1 + (mu[(x == 1) & (y == 0)] * rho)
    corr[(x == 1) & (y == 1)] = 1 - rho
    
    prob_final = prob_poisson_x * prob_poisson_y * corr
    log_p = np.where(prob_final <= 0, -10000, np.log(prob_final))
    
    penalty = lambda_reg * (np.sum(alpha**2) + np.sum(beta**2))
    return -np.sum(log_p * pesos) + penalty

def constraint_func(x):
    return sum(x[:n_selecoes])
cons = [{'type': 'eq', 'fun': constraint_func}]

# Smart Start
init_alpha = np.zeros(n_selecoes)
init_beta = np.zeros(n_selecoes)
for sel, idx in sel_idx.items():
    if sel in ataque_ingenuo:
        init_alpha[idx] = np.log(max(ataque_ingenuo[sel], 0.01))
        init_beta[idx] = np.log(max(defesa_ingenua[sel], 0.01))

init_alpha = init_alpha - np.mean(init_alpha)
init_params = np.concatenate([init_alpha, init_beta, [0.0], [0.2]])

bounds = [(-4, 4)] * (2 * n_selecoes) + [(-0.5, 0.5)] + [(0, 1.5)]

print("Iniciando otimização do Dixon-Coles...")
res = minimize(dixon_coles_loglik_vectorized, init_params, args=(df_opt, xi), 
               method='SLSQP', constraints=cons, bounds=bounds,
               options={'maxiter': 50})

alpha_opt = res.x[:n_selecoes]
beta_opt = res.x[n_selecoes:2*n_selecoes]
rho_opt = res.x[2*n_selecoes]

ataque_dc = {}
defesa_dc = {}
for sel in selecoes_copa:
    if sel in sel_idx:
        ataque_dc[sel] = np.exp(alpha_opt[sel_idx[sel]])
        defesa_dc[sel] = np.exp(beta_opt[sel_idx[sel]])
    else:
        ataque_dc[sel] = 1.0
        defesa_dc[sel] = 1.0

# 4. Escrever tudo em um CSV (Novo Banco de Dados)
dados_export = []
for sel in selecoes_copa:
    dados_export.append({
        'Selecao': sel,
        'Ataque_Ingenuo': ataque_ingenuo[sel],
        'Defesa_Ingenua': defesa_ingenua[sel],
        'Ataque_DC': ataque_dc[sel],
        'Defesa_DC': defesa_dc[sel]
    })

df_parametros = pd.DataFrame(dados_export)
df_parametros.to_csv('parametros_copa_2026.csv', index=False)
print("Sucesso! Parâmetros calculados e exportados para 'parametros_copa_2026.csv'")

# Salvar metadata auxiliar (rho e media)
with open('meta_dc.json', 'w') as f:
    json.dump({'rho': float(rho_opt), 'media_gols_geral': float(media_gols_geral)}, f)



---
## Preparando para a Simulação
Agora que já rodamos o modelo pesado, basta ler os parâmetros do CSV que criamos! Isso poupa muito tempo nas próximas vezes.


In [ ]:
# Lendo o Banco de Dados Novo
df_params = pd.read_csv('parametros_copa_2026.csv')

with open('meta_dc.json', 'r') as f:
    meta = json.load(f)

rho = meta['rho']
media_gols_geral = meta['media_gols_geral']

# Montando os 12 grupos de 4 seleções (Chaveamento da Copa)
grupos = {
    'Grupo A': ['Mexico', 'South Korea', 'Czech Republic', 'South Africa'],
    'Grupo B': ['Canada', 'Switzerland', 'Bosnia and Herzegovina', 'Qatar'],
    'Grupo C': ['Brazil', 'Morocco', 'Scotland', 'Haiti'],
    'Grupo D': ['United States', 'Australia', 'Paraguay', 'Turkey'],
    'Grupo E': ['Germany', 'Ivory Coast', 'Ecuador', 'Curaçao'],
    'Grupo F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'Grupo G': ['Egypt', 'Iran', 'Belgium', 'New Zealand'],
    'Grupo H': ['Spain', 'Uruguay', 'Cape Verde', 'Saudi Arabia'],
    'Grupo I': ['France', 'Norway', 'Senegal', 'Iraq'],
    'Grupo J': ['Argentina', 'Austria', 'Algeria', 'Jordan'],
    'Grupo K': ['Portugal', 'Colombia', 'DR Congo', 'Uzbekistan'],
    'Grupo L': ['England', 'Ghana', 'Panama', 'Croatia']
}

print("Grupos montados e parâmetros carregados. Prontos para a simulação!")



---
## Simulação da Fase de Grupos
Aqui simulamos cada um dos jogos dos grupos, listando o placar mais provável e a chance de vitória de cada equipe.


In [ ]:
def rho_correction(x, y, lambda_x, mu_y, rho):
    if x == 0 and y == 0: return 1 - (lambda_x * mu_y * rho)
    elif x == 0 and y == 1: return 1 + (lambda_x * rho)
    elif x == 1 and y == 0: return 1 + (mu_y * rho)
    elif x == 1 and y == 1: return 1 - rho
    return 1.0

def simular_jogo(time_a, time_b, df_params, rho, max_gols=6, modelo='DC'):
    # Pega os parâmetros do dataframe carregado
    params_a = df_params[df_params['Selecao'] == time_a].iloc[0]
    params_b = df_params[df_params['Selecao'] == time_b].iloc[0]
    
    if modelo == 'Ingenuo':
        lam_A = params_a['Ataque_Ingenuo'] * params_b['Defesa_Ingenua'] * media_gols_geral
        lam_B = params_b['Ataque_Ingenuo'] * params_a['Defesa_Ingenua'] * media_gols_geral
    else:
        # Modelo Dixon-Coles já está exp() no CSV
        lam_A = params_a['Ataque_DC'] * params_b['Defesa_DC']
        lam_B = params_b['Ataque_DC'] * params_a['Defesa_DC']
        
    matriz = np.zeros((max_gols, max_gols))
    for i in range(max_gols):
        for j in range(max_gols):
            prob = poisson.pmf(i, lam_A) * poisson.pmf(j, lam_B)
            if modelo == 'DC':
                prob *= rho_correction(i, j, lam_A, lam_B, rho)
            matriz[i, j] = prob
            
    matriz = matriz / matriz.sum()
    
    indice_max = np.unravel_index(np.argmax(matriz), matriz.shape)
    gols_A_max, gols_B_max = indice_max
    
    prob_vitoria_A = np.tril(matriz, -1).sum() * 100
    prob_vitoria_B = np.triu(matriz, 1).sum() * 100
    prob_empate = np.diag(matriz).sum() * 100
    
    return {
        'placar': f"{time_a} {gols_A_max} x {gols_B_max} {time_b}",
        'prob_A': prob_vitoria_A,
        'prob_B': prob_vitoria_B,
        'prob_Empate': prob_empate,
        'placar_mais_provavel_prob': matriz[indice_max] * 100
    }

print("=" * 60)
print(" RESULTADOS DA FASE DE GRUPOS DA COPA DO MUNDO 2026")
print("=" * 60)
print("Usando o Modelo Dixon-Coles para as simulações\n")

for grupo, times in grupos.items():
    print(f"🏆 {grupo.upper()} 🏆")
    for i in range(len(times)):
        for j in range(i + 1, len(times)):
            time_a = times[i]
            time_b = times[j]
            res = simular_jogo(time_a, time_b, df_params, rho, modelo='DC')
            
            print(f"⚽ {time_a} vs {time_b}")
            print(f"   ► Placar Sugerido: {res['placar']} ({res['placar_mais_provavel_prob']:.1f}% de chance do placar exato)")
            print(f"   ► Prognóstico: Vitória {time_a} ({res['prob_A']:.1f}%) | Empate ({res['prob_Empate']:.1f}%) | Vitória {time_b} ({res['prob_B']:.1f}%)")
            print("-" * 50)
    print("\n")



---
## Simulação Específica: Brasil vs Todas as Seleções
Nesta seção, comparamos o placar mais provável e as probabilidades para o Brasil enfrentando **todas as outras seleções** da Copa, utilizando tanto o modelo **Ingênuo** quanto o modelo **Dixon-Coles**.

In [4]:
time_foco = 'Brazil'
outras_selecoes = [sel for sel in df_params['Selecao'] if sel != time_foco]

print("=" * 85)
print(f" 🇧🇷 PALPITES E PROBABILIDADES: {time_foco.upper()} VS O MUNDO 🌍")
print("=" * 85 + "\n")

for adversario in outras_selecoes:
    res_ingenuo = simular_jogo(time_foco, adversario, df_params, rho, modelo='Ingenuo')
    res_dc = simular_jogo(time_foco, adversario, df_params, rho, modelo='DC')
    
    print(f"⚽ Confronto: {time_foco} vs {adversario}")
    print(f"{'Métrica':<15} | {'Modelo Ingênuo (Poisson)':<30} | {'Modelo Dixon-Coles':<30}")
    print("-" * 85)
    print(f"{'Placar':<15} | {res_ingenuo['placar']:<30} | {res_dc['placar']:<30}")
    print(f"{'Prob. Exata':<15} | {res_ingenuo['placar_mais_provavel_prob']:.1f}%{'':<25} | {res_dc['placar_mais_provavel_prob']:.1f}%")
    print(f"{'Vitória BR':<15} | {res_ingenuo['prob_A']:.1f}%{'':<25} | {res_dc['prob_A']:.1f}%")
    print(f"{'Empate':<15} | {res_ingenuo['prob_Empate']:.1f}%{'':<25} | {res_dc['prob_Empate']:.1f}%")
    print(f"{'Derrota BR':<15} | {res_ingenuo['prob_B']:.1f}%{'':<25} | {res_dc['prob_B']:.1f}%")
    print("\n")


 🇧🇷 PALPITES E PROBABILIDADES: BRAZIL VS O MUNDO 🌍

⚽ Confronto: Brazil vs Mexico
Métrica         | Modelo Ingênuo (Poisson)       | Modelo Dixon-Coles            
-------------------------------------------------------------------------------------
Placar          | Brazil 1 x 1 Mexico            | Brazil 1 x 1 Mexico           
Prob. Exata     | 13.5%                          | 14.6%
Vitória BR      | 35.2%                          | 41.5%
Empate          | 30.2%                          | 31.5%
Derrota BR      | 34.6%                          | 27.1%


⚽ Confronto: Brazil vs South Korea
Métrica         | Modelo Ingênuo (Poisson)       | Modelo Dixon-Coles            
-------------------------------------------------------------------------------------
Placar          | Brazil 1 x 1 South Korea       | Brazil 2 x 0 South Korea      
Prob. Exata     | 12.5%                          | 11.5%
Vitória BR      | 35.0%                          | 63.1%
Empate          | 26.3%                

---
## Visualização: Mapas de Calor (Heatmaps)
Abaixo, geramos a matriz de probabilidade exata de gols para confrontos selecionados do Brasil contra potências mundiais e zebras, permitindo visualizar de forma clara a diferença na distribuição das probabilidades entre o Modelo Ingênuo e o Dixon-Coles.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo premium para os gráficos
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#1e1e1e", "figure.facecolor": "#121212", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})

def plotar_mapas(time_a, time_b, df_params, rho, max_gols=6):
    params_a = df_params[df_params['Selecao'] == time_a].iloc[0]
    params_b = df_params[df_params['Selecao'] == time_b].iloc[0]
    
    # Matriz Ingênua
    lam_A_ing = params_a['Ataque_Ingenuo'] * params_b['Defesa_Ingenua'] * media_gols_geral
    lam_B_ing = params_b['Ataque_Ingenuo'] * params_a['Defesa_Ingenua'] * media_gols_geral
    
    matriz_ing = np.zeros((max_gols, max_gols))
    for i in range(max_gols):
        for j in range(max_gols):
            matriz_ing[i, j] = poisson.pmf(i, lam_A_ing) * poisson.pmf(j, lam_B_ing)
    matriz_ing = matriz_ing / matriz_ing.sum()
    
    # Matriz Dixon-Coles
    lam_A_dc = params_a['Ataque_DC'] * params_b['Defesa_DC']
    lam_B_dc = params_b['Ataque_DC'] * params_a['Defesa_DC']
    
    matriz_dc = np.zeros((max_gols, max_gols))
    for i in range(max_gols):
        for j in range(max_gols):
            prob = poisson.pmf(i, lam_A_dc) * poisson.pmf(j, lam_B_dc)
            prob *= rho_correction(i, j, lam_A_dc, lam_B_dc, rho)
            matriz_dc[i, j] = prob
    matriz_dc = matriz_dc / matriz_dc.sum()
    
    # Plotagem
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.heatmap(matriz_ing, annot=True, fmt='.2%', cmap='magma', cbar=False, 
                ax=axes[0], xticklabels=range(max_gols), yticklabels=range(max_gols))
    axes[0].set_title(f"Ingênuo (Poisson)\n{time_a} vs {time_b}")
    axes[0].set_xlabel(f"Gols {time_b}")
    axes[0].set_ylabel(f"Gols {time_a}")
    
    sns.heatmap(matriz_dc, annot=True, fmt='.2%', cmap='viridis', cbar=False, 
                ax=axes[1], xticklabels=range(max_gols), yticklabels=range(max_gols))
    axes[1].set_title(f"Dixon-Coles Dinâmico\n{time_a} vs {time_b}")
    axes[1].set_xlabel(f"Gols {time_b}")
    axes[1].set_ylabel(f"Gols {time_a}")
    
    plt.tight_layout()
    plt.show()

adversarios_plot = ['France', 'England', 'Spain', 'Haiti', 'Morocco', 'Scotland', 'Portugal', 'Argentina']
time_foco = 'Brazil'

for adv in adversarios_plot:
    if adv in df_params['Selecao'].values:
        plotar_mapas(time_foco, adv, df_params, rho)
    else:
        print(f"Seleção {adv} não encontrada nos parâmetros.")
